# factorlib demo — 四種因子類別 × 完整 API 展示

這份 notebook 不是教學文檔，是**可執行的功能索引**：每個 cell 對應 `factorlib/README.md` 的一段 API，照順序跑完就看過 factorlib 全部功能。

涵蓋：
- 四種 `factor_type`：`cross_sectional` / `event_signal` / `macro_panel` / `macro_common`
- 六個使用層級（Level 0–6）：單因子 → 自訂 config → 個別 metrics → batch + BHY → redundancy → charts → MLflow
- 資料：`tw_stock_daily_2017_2025.parquet`（TW 全市場日線），切出 2023-01–2024-06 當 demo 視窗。

## 0. Setup

In [1]:
from __future__ import annotations

import polars as pl
import factorlib as fl

pl.Config.set_tbl_rows(8)
pl.Config.set_fmt_str_lengths(60)
print('factorlib version:', getattr(fl, '__version__', 'dev'))

factorlib version: dev


In [2]:
# 載入原始 TW 日線，用 fl.adapt 把 user 欄位名對應到 factorlib canonical names
# (canonical = date / asset_id / price)
raw = pl.read_parquet('../tw_stock_daily_2017_2025.parquet')
raw_demo = fl.adapt(raw, date='date', asset_id='ticker', price='close_adj')

# validation schema 要求 date 為 Datetime(ms)，原始是 pl.Date → 轉型
raw_demo = raw_demo.with_columns(pl.col('date').cast(pl.Datetime('ms')))

# 切出 2023-01 ~ 2024-06 demo 視窗（控制 runtime）
raw_demo = raw_demo.filter(
    (pl.col('date') >= pl.datetime(2023, 1, 1))
    & (pl.col('date') < pl.datetime(2024, 7, 1))
)
print('rows:', raw_demo.height, '| assets:', raw_demo['asset_id'].n_unique())
raw_demo.head(3)

rows: 639352 | assets: 1837


asset_id,date,open_adj,high_adj,low_adj,price,volume,market,market_cap,twse_ind
str,datetime[ms],f64,f64,f64,f64,i64,str,f64,str
"""1101""",2023-01-03 00:00:00,30.9445,31.0827,30.6683,30.8524,11654098,"""TWSE""",239732.0,"""水泥工業"""
"""1102""",2023-01-03 00:00:00,34.9668,34.9668,34.4545,34.668,3878899,"""TWSE""",143965.0,"""水泥工業"""
"""1103""",2023-01-03 00:00:00,15.6454,15.6906,15.6002,15.6906,88117,"""TWSE""",13442.0,"""水泥工業"""


## 1. 四種因子類別總覽

`fl.describe_factor_types()` 印出所有註冊的 factor_type 與用途；
`fl.describe_profile(<type>)` 反射對應 Profile dataclass 的欄位、canonical p 與方法。
這兩個是「不用開檔就知道 factorlib 現在長什麼樣」的入口。

In [3]:
fl.describe_factor_types()
print()
print('list_factor_types ->', fl.list_factor_types())

  cross_sectional     : 截面因子（每期每資產有 signal，N ≥ 30）→ 選股
  event_signal        : 事件訊號（離散觸發）→ 事件交易
  macro_panel         : 宏觀 panel（小截面 N < 30）→ 跨國配置
  macro_common        : 宏觀共用（單一時序）→ 風險歸因

list_factor_types -> ['cross_sectional', 'event_signal', 'macro_panel', 'macro_common']


In [4]:
for ft in fl.list_factor_types():
    fl.describe_profile(ft)


  cross_sectional — CrossSectionalProfile
  ──────────────────────────────────────────────────
  Typed profile for a cross-sectional factor.

  Fields:
    factor_name                  str
    n_periods                    int
    ic_mean                      float
    ic_tstat                     float
    ic_p                         PValue
    ic_ir                        float
    hit_rate                     float
    hit_rate_p                   PValue
    ic_trend                     float
    ic_trend_p                   PValue
    monotonicity                 float
    q1_q5_spread                 float
    spread_tstat                 float
    spread_p                     PValue
    q1_concentration             float
    q1_concentration_eff_ratio   float
    oos_survival_ratio           float
    oos_sign_flipped             bool
    median_universe_n            int
    orthogonalize_applied        bool
    turnover                     float
    breakeven_cost              

## 2. Cross-sectional（選股因子）

訊號型態：每期每資產有連續值。典型用法 = 動量 / 價值 / 規模。
Canonical test: `ic_p`（IC 非重疊 t-test）。

### 2.1 Build factor（用 factorlib 內建 generator）

In [5]:
from factorlib.factors import generate_momentum, generate_momentum_60d
from factorlib.factors.volatility import generate_volatility

mom20 = generate_momentum(raw_demo, lookback=20)
print('columns:', mom20.columns)
print('shape:', mom20.shape)
mom20.select(['asset_id', 'date', 'price', 'factor']).head(3)

columns: ['asset_id', 'date', 'open_adj', 'high_adj', 'low_adj', 'price', 'volume', 'market', 'market_cap', 'twse_ind', 'factor']
shape: (602684, 11)


asset_id,date,price,factor
str,datetime[ms],f64,f64
"""1101""",2023-02-10 00:00:00,33.9377,0.100002
"""1101""",2023-02-13 00:00:00,34.0758,0.104478
"""1101""",2023-02-14 00:00:00,34.3061,0.081277


### 2.2 Level 0 — 最簡單 `fl.evaluate`

一行跑完 preprocess + all metrics + diagnostics，回傳 `CrossSectionalProfile`（frozen dataclass）。

In [6]:
profile = fl.evaluate(mom20, 'Mom_20D', factor_type='cross_sectional')

print('type:       ', type(profile).__name__)
print('verdict:    ', profile.verdict())
print('canonical_p:', f'{profile.canonical_p:.4f}')
print('ic_mean:    ', f'{profile.ic_mean:+.4f}')
print('ic_tstat:   ', f'{profile.ic_tstat:+.2f}')
print('ic_ir:      ', f'{profile.ic_ir:+.3f}')
print('q1_q5 spread:', f'{profile.q1_q5_spread:+.4f}')
print('turnover:   ', f'{profile.turnover:.2%}')
print('net_spread: ', f'{profile.net_spread:+.4f}')

type:        CrossSectionalProfile
verdict:     FAILED
canonical_p: 0.7251
ic_mean:     -0.0098
ic_tstat:    -0.35
ic_ir:       -0.103
q1_q5 spread: +0.0009
turnover:    7.34%
net_spread:  +0.0005


In [7]:
# profile.diagnose() 回傳結構化 Diagnostic — 包含 severity / code / message
for d in profile.diagnose():
    print(f'[{d.severity:<7s}] {d.code:<35s} {d.message}')

[info   ] cs.orthogonalize_not_applied        orthogonalize=False — factor exposures were not regressed against the market's standard risk factors (size / value / momentum / industry). Any alpha observed here may be a repackaging of a known risk premium.
[warn   ] cs.ic_weak_spread_strong            IC not significant (p > 0.05) but Q1-Q5 spread is — alpha may be driven by outlier stocks rather than the broad cross-section.
[veto   ] cs.oos_sign_flipped                 OOS sample shows a sign flip vs IS — overfitting risk; the factor did not generalize.


### 2.3 Level 1 — 自訂 `CrossSectionalConfig`

切換 forward horizon、quantile groups、trading-cost 估計等。

In [8]:
cfg = fl.CrossSectionalConfig(
    forward_periods=10,            # 10 日 forward return
    n_groups=5,                    # 5 組 quantile
    q_top=0.2,                     # Q1 取前 20%
    orthogonalize=False,
    mad_n=3.0,
    return_clip_pct=(0.01, 0.99),
    estimated_cost_bps=30,
)
profile_10d = fl.evaluate(mom20, 'Mom_20D_h10', config=cfg)
print('verdict @10d:', profile_10d.verdict(), '| ic_ir:', f'{profile_10d.ic_ir:+.3f}')

verdict @10d: FAILED | ic_ir: -0.044


### 2.4 Level 2 — 個別 metrics（繞過 Profile）

當你只想算某個統計量、或要餵 metric 進自己的 pipeline，不需要整個 Profile。
流程：`fl.preprocess` → 個別 `metrics.*` 函式（全部回 `MetricOutput`）。

In [9]:
from factorlib.metrics import (
    compute_ic, ic, ic_ir, quantile_spread, monotonicity,
)

prepared = fl.preprocess(mom20, config=cfg)
print('prepared columns:', prepared.columns, '\n')

ic_series = compute_ic(prepared)  # DataFrame[date, ic]
ic_m = ic(ic_series, forward_periods=cfg.forward_periods)
ir_m = ic_ir(ic_series)
spread_m = quantile_spread(prepared, forward_periods=cfg.forward_periods, n_groups=cfg.n_groups)
mono_m = monotonicity(prepared, forward_periods=cfg.forward_periods, n_groups=cfg.n_groups)

# MetricOutput 的 p-value 放在 .metadata['p_value']，__repr__ 會印摘要
for name, out in [('ic', ic_m), ('ic_ir', ir_m), ('q1_q5_spread', spread_m), ('monotonicity', mono_m)]:
    p = out.metadata.get('p_value')
    p_str = 'n/a' if p is None else f'{p:.4f}'
    stat_str = 'n/a' if out.stat is None else f'{out.stat:+.2f}'
    print(f'{name:<15s} value={out.value:+.4f}  stat={stat_str:>6s}  p={p_str}  sig={out.significance or ""}')

prepared columns: ['date', 'asset_id', 'factor_raw', 'factor', 'forward_return', 'abnormal_return', 'price'] 

ic              value=-0.0037  stat= -0.33  p=0.7452  sig=
ic_ir           value=-0.0441  stat=   n/a  p=n/a  sig=
q1_q5_spread    value=+0.0007  stat= +2.37  p=0.0238  sig=**
monotonicity    value=+0.6212  stat= +1.69  p=0.1017  sig=


### 2.5 Level 3 — `evaluate_batch` + BHY + rank + top

批次評估多因子，回傳 polars-native `ProfileSet`。
`multiple_testing_correct` 做 BHY 多重檢定校正；`filter` / `rank_by` / `top` 可鏈式串接。

In [10]:
# 準備三個候選 CS 因子
mom60 = generate_momentum_60d(raw_demo)
vol20 = generate_volatility(raw_demo, lookback=20)

factors_map = {
    'Mom_20D': mom20,
    'Mom_60_5': mom60,
    'Vol_20D': vol20,
}
ps = fl.evaluate_batch(factors_map, factor_type='cross_sectional')
print('ProfileSet size:', len(ps), '| profile class:', ps.profile_cls.__name__)
ps.to_polars().select(['factor_name', 'ic_mean', 'ic_ir', 'q1_q5_spread', 'canonical_p'])  # quick glance


ProfileSet size: 3 | profile class: CrossSectionalProfile


factor_name,ic_mean,ic_ir,q1_q5_spread,canonical_p
str,f64,f64,f64,f64
"""Mom_20D""",-0.009785,-0.102516,0.00091,0.725113
"""Mom_60_5""",0.011486,0.124206,0.000762,0.401779
"""Vol_20D""",0.048925,0.310812,-0.000532,0.01565


In [11]:
# BHY 多重檢定 + 按 IC_IR 排序 + 取 top 2
top = (
    ps
    .multiple_testing_correct(p_source='canonical_p', fdr=0.10)
    .rank_by('ic_ir', descending=True)
    .top(2)
)
top.to_polars().select([
    'factor_name', 'ic_p', 'p_adjusted', 'bhy_significant', 'ic_ir',
])

factor_name,ic_p,p_adjusted,bhy_significant,ic_ir
str,f64,f64,bool,f64
"""Vol_20D""",0.01565,0.086077,true,0.310812
"""Mom_60_5""",0.401779,1.0,false,0.124206


In [12]:
# 也能傳 pl.Expr 或 Callable[[Profile], bool] 給 filter
stable = ps.filter(pl.col('ic_ir').abs() > 0.1)
print('factors with |IC_IR| > 0.1:', [p.factor_name for p in stable])

factors with |IC_IR| > 0.1: ['Mom_20D', 'Mom_60_5', 'Vol_20D']


### 2.6 Level 4 — Redundancy matrix

多因子之間的成對 `|ρ|`，抓出冗餘訊號。

In [13]:
# redundancy_matrix 需要 per-factor Artifacts；evaluate_batch 不保留，
# 要自己 loop fl.evaluate 再 build_artifacts 並收集。
from factorlib.evaluation.pipeline import build_artifacts

arts: dict[str, object] = {}
for name, fdf in factors_map.items():
    prep = fl.preprocess(fdf, config=fl.CrossSectionalConfig())
    a = build_artifacts(prep, fl.CrossSectionalConfig())
    a.factor_name = name
    arts[name] = a

redund = fl.redundancy_matrix(ps, method='value_series', artifacts=arts)
redund

/Users/cfh00884862/Desktop/dst/code/factor-analysis/factorlib/metrics/redundancy.py:114: UserWarning: redundancy_matrix: factors ['Mom_60_5'] have missing dates that other factors cover; correlation is computed on the intersection. If that shrinks the window unacceptably, drop the short-history factors from the ProfileSet.
  return _value_series_matrix(names, artifacts, profiles.profile_cls)


factor,Mom_20D,Mom_60_5,Vol_20D
str,f64,f64,f64
"""Mom_20D""",1.0,0.495109,0.118649
"""Mom_60_5""",0.495109,1.0,0.484736
"""Vol_20D""",0.118649,0.484736,1.0


### 2.7 Level 5 — Charts (optional dep: `factorlib[charts]`)

`build_artifacts` 保留中間結果（IC series / spread series / quantile group returns…），
丟進 `report_charts` 產 plotly 圖。未安裝 `plotly` 會 raise ImportError — 此處 try/except 包起來。

In [14]:
from factorlib.evaluation.pipeline import build_artifacts

prepared = fl.preprocess(mom20, config=fl.CrossSectionalConfig())
artifacts = build_artifacts(prepared, fl.CrossSectionalConfig())
artifacts.factor_name = 'Mom_20D'
print('artifacts.intermediates keys:', sorted(artifacts.intermediates.keys()))

try:
    from factorlib.charts import report_charts
    figs = report_charts(artifacts)
    print(f'produced {len(figs)} figures:', list(figs)[:5])
    # 在 notebook 直接顯示第一張
    next(iter(figs.values())).show()
except ImportError as e:
    print('charts skipped — install with `pip install factorlib[charts]`:', e)

artifacts.intermediates keys: ['ic_series', 'ic_values', 'spread_series']
produced 4 figures: ['cumulative_ic', 'ic_distribution', 'spread_ts', 'quantile_returns']


### 2.8 Level 6 — MLflow tracking (optional dep: `factorlib[mlflow]`)

`on_result` callback 在 `evaluate_batch` 的每個因子算完後觸發，
可用來 log profile 到 MLflow experiment。這裡只示範寫法，不實跑（避免副作用）。

In [15]:
# 不實跑 — 只展示 API 形狀
demo = '''
from factorlib.integrations.mlflow import FactorTracker

tracker = FactorTracker('Factor_Zoo')
ps = fl.evaluate_batch(
    factors_map,
    factor_type='cross_sectional',
    on_result=lambda name, p: tracker.log_profile(p, factor_type='cross_sectional'),
)
'''
print(demo)


from factorlib.integrations.mlflow import FactorTracker

tracker = FactorTracker('Factor_Zoo')
ps = fl.evaluate_batch(
    factors_map,
    factor_type='cross_sectional',
    on_result=lambda name, p: tracker.log_profile(p, factor_type='cross_sectional'),
)



## 3. Event-signal（事件交易因子）

訊號型態：離散觸發 `{-1, 0, +1}`。Canonical test: `caar_p`（CAAR 非重疊 t-test）。
範例：黃金交叉 / 死亡交叉 → 事件後 5 日有異常報酬嗎？

### 3.1 Build golden/death-cross event signal

In [16]:
event_df = (
    raw_demo.sort(['asset_id', 'date'])
    .with_columns(
        pl.col('price').rolling_mean(5).over('asset_id').alias('ma5'),
        pl.col('price').rolling_mean(20).over('asset_id').alias('ma20'),
    )
    .with_columns(
        pl.when(pl.col('ma5') > pl.col('ma20')).then(1)
          .otherwise(-1).alias('cross_state')
    )
    .with_columns(
        (pl.col('cross_state') - pl.col('cross_state').shift(1).over('asset_id')).alias('delta')
    )
    .with_columns(
        pl.when(pl.col('delta') == 2).then(1.0)   # 黃金交叉
        .when(pl.col('delta') == -2).then(-1.0)   # 死亡交叉
        .otherwise(0.0).alias('factor')
    )
    .filter(pl.col('factor').is_not_null() & pl.col('ma20').is_not_null())
    .select(['asset_id', 'date', 'price', 'factor'])
)
print('factor value counts:')
print(event_df['factor'].value_counts().sort('factor'))

factor value counts:
shape: (3, 2)
┌────────┬────────┐
│ factor ┆ count  │
│ ---    ┆ ---    │
│ f64    ┆ u32    │
╞════════╪════════╡
│ -1.0   ┆ 19738  │
│ 0.0    ┆ 564016 │
│ 1.0    ┆ 20760  │
└────────┴────────┘


### 3.2 evaluate（自動切到 `EventProfile`）

In [17]:
ev_cfg = fl.EventConfig(
    forward_periods=5,
    event_window_pre=5,
    event_window_post=20,
    cluster_window=3,
    adjust_clustering='none',
)
ev_profile = fl.evaluate(event_df, 'GoldenCross', config=ev_cfg)

print('type:           ', type(ev_profile).__name__)
print('n_events:       ', ev_profile.n_events)
print('n_periods (dates):', ev_profile.n_periods)
print('verdict:        ', ev_profile.verdict())
print('CAAR mean:      ', f'{ev_profile.caar_mean:+.4f}')
print('CAAR p (canonical):', f'{ev_profile.caar_p:.4f}')
print('BMP z:          ', f'{ev_profile.bmp_zstat:+.2f}  p={ev_profile.bmp_p:.4f}')
print('event_hit_rate: ', f'{ev_profile.event_hit_rate:.2%}  p={ev_profile.event_hit_rate_p:.4f}')
print('profit_factor:  ', f'{ev_profile.profit_factor:.3f}')
print('clustering_hhi (normalized):', ev_profile.clustering_hhi_normalized)

type:            EventProfile
n_events:        39775
n_periods (dates): 331
verdict:         FAILED
CAAR mean:       -0.0000
CAAR p (canonical): 0.4363
BMP z:           +1.22  p=0.2234
event_hit_rate:  49.37%  p=0.0113
profit_factor:   0.993
clustering_hhi (normalized): 0.0015179888788612276


In [18]:
for d in ev_profile.diagnose():
    print(f'[{d.severity:<7s}] {d.code:<35s} {d.message}')

[veto   ] event.caar_bmp_sign_mismatch        CAAR and BMP have opposite signs — CAAR may be spuriously inflated by event-induced variance; use BMP as the trustworthy signal.
[warn   ] event.hit_rate_only                 Hit-rate is significant but CAAR and BMP are not — direction is correct on average, but economic magnitude is weak.
[warn   ] event.caar_trend_decay              CAAR trend is significantly negative — signal may be decaying.
[veto   ] event.oos_sign_flipped              OOS sample shows a sign flip vs IS — overfitting risk.


### 3.3 Level 2 — individual event metrics

In [19]:
from factorlib.metrics import (
    compute_caar, caar, bmp_test, event_hit_rate,
    corrado_rank_test, clustering_diagnostic,
)

ev_prepared = fl.preprocess(event_df, config=ev_cfg)

caar_series = compute_caar(ev_prepared)   # per-event-date signed AR
print('caar:         ', caar(caar_series, forward_periods=5))
print('BMP:          ', bmp_test(ev_prepared, forward_periods=5))
print('event_hit_rate:', event_hit_rate(ev_prepared))
print('corrado rank: ', corrado_rank_test(ev_prepared))
print('clustering:   ', clustering_diagnostic(ev_prepared, cluster_window=3))

caar:          MetricOutput(caar=-0.0002, stat=-0.33)
BMP:           MetricOutput(bmp_sar=-0.0249, stat=-4.14, sig=***)
event_hit_rate: MetricOutput(event_hit_rate=0.4548, stat=-18.05, sig=***)
corrado rank:  MetricOutput(corrado_rank=-0.0136, stat=-9.41, sig=***)
clustering:    MetricOutput(clustering_hhi=0.0045)


## 4. Macro panel（跨國/小截面配置因子）

訊號型態：連續值 + 小截面 `N < 30`。典型用法 = 跨國 CPI / 利差配置。
Canonical test: `fm_beta_p`（Fama-MacBeth Newey-West t-test）。

由於 TW 資料只有單一市場，這裡用 **TWSE 產業分類** 當 pseudo-country（N≈30）。

### 4.1 Build industry-aggregated panel

In [20]:
industry = (
    raw_demo.group_by(['date', 'twse_ind'])
    .agg(pl.col('price').mean())
    .rename({'twse_ind': 'asset_id'})
    .drop_nulls(['asset_id'])
    .sort(['asset_id', 'date'])
)
# factor = 產業相對 60 日均價偏離（簡單的 mean-reversion proxy）
industry = industry.with_columns(
    (pl.col('price') / pl.col('price').rolling_mean(60).over('asset_id') - 1).alias('factor')
).drop_nulls('factor')
print('industry N:', industry['asset_id'].n_unique())
industry.head(3)

industry N: 34


date,asset_id,price,factor
datetime[ms],str,f64,f64
2023-04-13 00:00:00,"""光電業""",57.617211,0.041827
2023-04-14 00:00:00,"""光電業""",57.151046,0.031346
2023-04-17 00:00:00,"""光電業""",58.176238,0.047598


### 4.2 evaluate + `MacroPanelConfig`

In [21]:
mp_cfg = fl.MacroPanelConfig(
    forward_periods=5,
    n_groups=3,               # 小截面 → 少分組
    demean_cross_section=False,
    min_cross_section=10,
)
mp_profile = fl.evaluate(industry, 'IndRelValue', config=mp_cfg)

print('verdict:              ', mp_profile.verdict())
print('fm_beta_mean (λ):     ', f'{mp_profile.fm_beta_mean:+.5f}')
print('fm_beta_tstat:        ', f'{mp_profile.fm_beta_tstat:+.2f}')
print('fm_beta_p (canonical):', f'{mp_profile.fm_beta_p:.4f}')
print('pooled_beta:          ', f'{mp_profile.pooled_beta:+.5f}  p={mp_profile.pooled_beta_p:.4f}')
print('beta_sign_consistency:', f'{mp_profile.beta_sign_consistency:.2%}')
print('q1_q5_spread:         ', f'{mp_profile.q1_q5_spread:+.4f}')
print('median cross-section N:', mp_profile.median_cross_section_n)

verdict:               FAILED
fm_beta_mean (λ):      +0.00009
fm_beta_tstat:         +0.82
fm_beta_p (canonical): 0.4121
pooled_beta:           -0.00002  p=0.7891
beta_sign_consistency: 54.30%
q1_q5_spread:          +0.0002
median cross-section N: 34


## 5. Macro common（全市場共用因子）

訊號型態：單一時序，每個資產共用同一個 factor 值。典型用法 = VIX / 黃金 / USD index。
Canonical test: `ts_beta_p`（per-asset 時序 OLS β 的截面 t-test）。

範例：市場截面波動率（所有資產日報酬的 cross-sectional std）→ 對每檔股票的 exposure 穩定嗎？

### 5.1 Build market vol + broadcast 到每個資產

In [22]:
# 市場波動率（每日：跨資產日報酬 std）
mkt_vol = (
    raw_demo.sort(['asset_id', 'date'])
    .with_columns(pl.col('price').pct_change().over('asset_id').alias('ret'))
    .group_by('date').agg(pl.col('ret').std().alias('factor'))
    .sort('date').drop_nulls('factor')
)
print('market vol head:')
print(mkt_vol.head(3))

# 為了控 runtime，sample 50 檔標的
sample_assets = raw_demo.select('asset_id').unique().head(50)['asset_id'].to_list()
common_df = (
    raw_demo.filter(pl.col('asset_id').is_in(sample_assets))
    .select(['date', 'asset_id', 'price'])
    .join(mkt_vol, on='date', how='inner')
)
print('common panel shape:', common_df.shape)

market vol head:
shape: (3, 2)
┌─────────────────────┬──────────┐
│ date                ┆ factor   │
│ ---                 ┆ ---      │
│ datetime[ms]        ┆ f64      │
╞═════════════════════╪══════════╡
│ 2023-01-04 00:00:00 ┆ 0.018249 │
│ 2023-01-05 00:00:00 ┆ 0.016922 │
│ 2023-01-06 00:00:00 ┆ 0.016153 │
└─────────────────────┴──────────┘


common panel shape: (17388, 4)


### 5.2 evaluate + `MacroCommonConfig`

In [23]:
mc_cfg = fl.MacroCommonConfig(
    forward_periods=5,
    ts_window=60,
    tradable=False,
)
mc_profile = fl.evaluate(common_df, 'MktVol', config=mc_cfg)

print('verdict:              ', mc_profile.verdict())
print('n_assets:             ', mc_profile.n_assets)
print('ts_beta_mean:         ', f'{mc_profile.ts_beta_mean:+.5f}')
print('ts_beta_tstat:        ', f'{mc_profile.ts_beta_tstat:+.2f}')
print('ts_beta_p (canonical):', f'{mc_profile.ts_beta_p:.4f}')
print('mean R²:              ', f'{mc_profile.mean_r_squared:.4f}')
print('beta_sign_consistency:', f'{mc_profile.ts_beta_sign_consistency:.2%}')

verdict:               FAILED
n_assets:              50
ts_beta_mean:          -0.00037
ts_beta_tstat:         -1.97
ts_beta_p (canonical): 0.0539
mean R²:               0.0137
beta_sign_consistency: 64.00%


## 6. 切換因子類別的 cheat sheet

只要換 `factor_type=` 字串（或對應的 `XxxConfig`），同一組 `fl.evaluate` / `fl.evaluate_batch` / `ProfileSet` API 都能用——差別在回傳的 Profile dataclass 欄位不同、canonical p 對應的統計量不同。

In [24]:
cheat = pl.DataFrame({
    'factor_type': ['cross_sectional', 'event_signal', 'macro_panel', 'macro_common'],
    'signal_shape': ['每期每資產連續值', '離散 {-1,0,+1}', '連續值，小截面 N<30', '單一時序、全資產共用'],
    'Config':       ['CrossSectionalConfig', 'EventConfig', 'MacroPanelConfig', 'MacroCommonConfig'],
    'Profile':      ['CrossSectionalProfile', 'EventProfile', 'MacroPanelProfile', 'MacroCommonProfile'],
    'canonical_p':  ['ic_p', 'caar_p', 'fm_beta_p', 'ts_beta_p'],
    'core_question': ['排序能預測截面報酬差異嗎？', '事件後報酬有異常嗎？',
                       '宏觀指標能預測跨資產配置嗎？', '資產對共同因子的 exposure 穩定嗎？'],
})
cheat

factor_type,signal_shape,Config,Profile,canonical_p,core_question
str,str,str,str,str,str
"""cross_sectional""","""每期每資產連續值""","""CrossSectionalConfig""","""CrossSectionalProfile""","""ic_p""","""排序能預測截面報酬差異嗎？"""
"""event_signal""","""離散 {-1,0,+1}""","""EventConfig""","""EventProfile""","""caar_p""","""事件後報酬有異常嗎？"""
"""macro_panel""","""連續值，小截面 N<30""","""MacroPanelConfig""","""MacroPanelProfile""","""fm_beta_p""","""宏觀指標能預測跨資產配置嗎？"""
"""macro_common""","""單一時序、全資產共用""","""MacroCommonConfig""","""MacroCommonProfile""","""ts_beta_p""","""資產對共同因子的 exposure 穩定嗎？"""


---

## 附錄 A：全 factor_type 收到的 Profile 一覽

把本 notebook 跑出來的四個 profile 並排，展示不同 factor_type 的 verdict + canonical_p 是同一套介面。

In [25]:
summary = pl.DataFrame([
    {
        'factor_type': 'cross_sectional',
        'factor_name': profile.factor_name,
        'verdict': profile.verdict(),
        'canonical_p': profile.canonical_p,
        'n_periods': profile.n_periods,
    },
    {
        'factor_type': 'event_signal',
        'factor_name': ev_profile.factor_name,
        'verdict': ev_profile.verdict(),
        'canonical_p': ev_profile.canonical_p,
        'n_periods': ev_profile.n_periods,
    },
    {
        'factor_type': 'macro_panel',
        'factor_name': mp_profile.factor_name,
        'verdict': mp_profile.verdict(),
        'canonical_p': mp_profile.canonical_p,
        'n_periods': mp_profile.n_periods,
    },
    {
        'factor_type': 'macro_common',
        'factor_name': mc_profile.factor_name,
        'verdict': mc_profile.verdict(),
        'canonical_p': mc_profile.canonical_p,
        'n_periods': mc_profile.n_periods,
    },
])
summary

factor_type,factor_name,verdict,canonical_p,n_periods
str,str,str,f64,i64
"""cross_sectional""","""Mom_20D""","""FAILED""",0.725113,330
"""event_signal""","""GoldenCross""","""FAILED""",0.436297,331
"""macro_panel""","""IndRelValue""","""FAILED""",0.412085,291
"""macro_common""","""MktVol""","""FAILED""",0.053921,349
